In [ ]:
!nvidia-smi

In [ ]:
import os
import requests

url = r"https://arxiv.org/pdf/2412.19437"
pdf_path = os.path.join(os.getcwd(), "Data_file.pdf")

if not os.path.exists(pdf_path):
    response = requests.get(url)
    if response.status_code == 200:
        with open(pdf_path, "wb") as f:
            f.write(response.content)
        print(f"[INFO] the file is downloaded at : {pdf_path}")
else:
    print(f"[INFO] file exist : {pdf_path} ")


In [ ]:
import fitz  # PyMuPDF

def clean_text(text: str):
    text = text.replace("\n", " ").strip()
    return text

def open_pdf(file_path: str):
    document = fitz.open(file_path)
    pages_and_texts = []
    for page_num, page in enumerate(document):
        text = page.get_text()
        cleaned_text = clean_text(text)
        pages_and_texts.append({
            "text_of_page": cleaned_text,
            "page_count_char": len(cleaned_text),
            "page_count_word": len(cleaned_text.split(" ")),
            "page_count_sentences": len(cleaned_text.split(". ")),
            "tokens_num": len(cleaned_text) // 4,
            "page_number": page_num + 1
        })
    return pages_and_texts

pages_and_texts = open_pdf("Data_file.pdf")
pages_and_texts[3]


In [ ]:
import pandas as pd
df = pd.DataFrame(pages_and_texts)
df.head(10)


In [ ]:
df.describe().round(2)


# Turn text to sentences 

In [ ]:
from spacy.lang.en import English

nlp = English()
nlp.add_pipe("sentencizer")

semtemces = "this my name . hellow mr.ahmed . what is your name ?"
doc = nlp(semtemces)
list(doc.sents)


# Now try the `data` 

In [ ]:
for page in pages_and_texts:
    page["sentences"] = list(nlp(page["text_of_page"]).sents)
    page["sentences_num"] = len(page["sentences"])

for page in pages_and_texts:
    page["sentences"] = [str(s) for s in page["sentences"]]

pages_and_texts[5]


In [ ]:
pages_and_texts[5]["sentences"][6]


In [ ]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)


In [ ]:
test = list(range(1, 25))
slice_size = 10
x = [test[i:i+slice_size] for i in range(0, len(test), slice_size)]
x


In [ ]:
def chunker(input_list: list[str], slice_size: int = slice_size) -> list[list[str]]:
    return [input_list[i:i+slice_size] for i in range(0, len(input_list), slice_size)]

for page in pages_and_texts:
    page["sentences_chunk"] = chunker(page["sentences"])
    page["sentences_chunk_num"] = len(page["sentences_chunk"])

pages_and_texts[5]


In [ ]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)


In [ ]:
pages_and_chunk = []
for page in pages_and_texts:
    for chunk in page["sentences_chunk"]:
        chunk_dict = {}
        paragragh = "".join(chunk).replace("  ", " ").strip()
        chunk_dict["paragraphe"] = paragragh
        chunk_dict["word_count"] = len(paragragh.split(" "))
        chunk_dict["char_count"] = len(paragragh)
        chunk_dict["tokens_count"] = len(paragragh) // 4
        chunk_dict["sentence_count"] = len(chunk)
        chunk_dict["page_number"] = page["page_number"]
        pages_and_chunk.append(chunk_dict)

pages_and_chunk[5], len(pages_and_chunk), len(pages_and_texts)


In [ ]:
df = pd.DataFrame(pages_and_chunk)
df.describe().round(2)


In [ ]:
df[df["word_count"] < 4]


In [ ]:
filtered_chunks = [chunk for chunk in pages_and_chunk if chunk["tokens_count"] >= 30]
filtered_chunks[111]


In [ ]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


In [ ]:
test_set = ["my name", "her name", "the naem"]
embedding = model.encode(test_set)
embedding = dict(zip(test_set, embedding))
embedding, embedding["my name"].shape


In [ ]:
from tqdm import tqdm

for chunk in tqdm(pages_and_chunk):
    chunk["embedding"] = model.encode(chunk["paragraphe"], device="cuda")


In [ ]:
page_and_Embedding_df = pd.DataFrame(pages_and_chunk)
page_and_Embedding_file_name = "page_and_Embedding.csv"
page_and_Embedding_path = os.path.join(os.getcwd(), page_and_Embedding_file_name)
page_and_Embedding_df.to_csv(page_and_Embedding_path, index=False)


In [ ]:
pages_and_chunk = pd.read_csv(page_and_Embedding_path)
pages_and_chunk


In [ ]:
data = pages_and_chunk.to_dict(orient="records")
data[5]


In [ ]:
import torch
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
device


In [ ]:
for item in data:
    item['embedding'] = np.fromstring(item['embedding'].strip('[]'), sep=' ')


In [ ]:
embeddings = np.array([item["embedding"] for item in data])
embeddings_df = pd.DataFrame(embeddings)
print(embeddings_df.shape)  # should be (153, 384)


153 rows x 384 dimensions — one embedding vector per chunk.

In [ ]:
from time import perf_counter as timer

query = "attentions in models datas"
embeddings_tensor = torch.tensor(np.array(embeddings), dtype=torch.float32)
query_embedding = model.encode(query, device=device)

start = timer()
dot_product = util.dot_score(a=query_embedding, b=embeddings_tensor)[0]
end = timer()
print(f"[INFO]: time it take {(end-start):.5f} seconds")

dot_topk = torch.topk(dot_product, 3)
dot_topk


In [ ]:
# just test what idx.item() do
[idx.item() for idx in dot_topk.indices]

#it turn tensor to scaller 

In [ ]:
# Show the top-k matching paragraphs
for score, idx in zip(dot_topk.values, dot_topk.indices):
    print(f"Score: {score:.4f} | Page: {data[idx.item()]['page_number']}")
    print(data[idx.item()]['paragraphe'])
    print("-" * 120)


# functionalize the query search 

In [ ]:
def get_answer(query:str,
               embedings_data=embeddings,
               embedding_model: SentenceTransformer=model,
               top_k :int = 5,
               time_taken:bool =True):
    # embed the query
    query_embed= embedding_model.encode(query,convert_to_tensor=True)

    # make sure that data are in the same device (cuda) the GPU in this case
    embedings_data = torch.tensor(np.array(embedings_data), dtype=torch.float32).to('cuda')

    # compare the embedding of query with ather embeddings frome book/dataset
    start = timer()
    dot_product = util.dot_score(a=query_embed, b=embedings_data)[0]
    end = timer()    

    if time_taken:
        print(f"[INFO]time {(end-start):.5f}")
    top_results = torch.topk(dot_product,top_k)
    scores=[score.item() for score in top_results[0]]
    indeces =[score.item() for score in top_results[1]]
    # scores,indeces = top_results
    # print(f"{top_results}")
    return scores,indeces 

def print_top_Result(query:str,
               embedings_data=embeddings,
               embedding_model: SentenceTransformer=model,
               top_k :int = 5,
               time_taken:bool =True,
               pages_and_chunk=data):
    score,indcis=get_answer(query,embedings_data,embedding_model,top_k,time_taken)

    for s, i in zip(score, indcis):
        print(f"Score: {s:.4f} | Index: {i} | Page_number: {pages_and_chunk[i]['page_number']}\nChunk: {pages_and_chunk[i]["paragraphe"]}")
        print("-"*120)

In [ ]:
print_top_Result("github")

In [ ]:
# Get GPU available memory
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2 ** 30))
print(f"Available GPU memory: {gpu_memory_gb} GB ")